In [27]:
!pip install clustering-benchmarks -q

In [28]:
import clustbench
data_url = "https://github.com/gagolews/clustering-data-v1/raw/v1.1.0"

In [29]:
battery_datasets_dict = {
                         'fcps': ['atom', 'chainlink', 'engytime', 'hepta', 'lsun', 'target', 'tetra', 'twodiamonds', 'wingnut'],
                         'uci': ['ecoli', 'glass', 'ionosphere', 'sonar', 'statlog', 'wdbc', 'wine', 'yeast'],
                         #'mnist': ['digits', 'fashion'], # Genie digits ~ 32 min
                         #'sipu': ['worms_64'], # Genie ~ 9 min
                         }

## Create autoencoder using pytorch

In [30]:
import torch
import torch.nn as nn
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

def autoencoder_feat_eng(X, n_embeddings=8, sparsity_penalty=0.9):

  X = StandardScaler().fit_transform(X)
  X = torch.tensor(X, dtype=torch.float)

  # --- Autoencoder definition ---
  class Autoencoder(nn.Module):
      def __init__(self, input_dim, latent_dim=n_embeddings):
          super().__init__()
          self.encoder = nn.Sequential(
              nn.Linear(input_dim, 64),
              nn.ReLU(),
              nn.Linear(64, latent_dim)
          )
          self.decoder = nn.Sequential(
              nn.Linear(latent_dim, 64),
              nn.ReLU(),
              nn.Linear(64, input_dim)
          )

      def forward(self, x):
          z = self.encoder(x)
          x_hat = self.decoder(z)
          return x_hat, z

  # --- Train AE ---
  input_dim = X.shape[1] # Dynamically set input_dim
  model = Autoencoder(input_dim)
  optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
  criterion = nn.MSELoss()

  for epoch in range(100):
      x_hat, z = model(X)
      # Calculate reconstruction loss
      reconstruction_loss = criterion(x_hat, X)
      # Calculate L1 sparsity penalty
      l1_loss = torch.norm(z, p=1)
      # Total loss with sparsity penalty
      loss = reconstruction_loss + sparsity_penalty * l1_loss

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

  # --- Cluster latent space ---
  with torch.no_grad():
      latent = model.encoder(X).numpy()

  return latent

### Difference between original dataset and embeddings from the autoencoder

In [31]:
data_url = "https://github.com/gagolews/clustering-data-v1/raw/v1.1.0"
battery = "uci"
dataset = "statlog"
b = clustbench.load_dataset(battery, dataset, url=data_url)

In [32]:
import pandas as pd
pd.DataFrame(b.data)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,1.765321,1.035128,0.001834,-0.000089,-0.020113,-0.097886,-0.024912,-0.146014,0.428177,0.372141,0.588547,0.323848,-0.168114,0.481106,-0.312993,0.570538,-0.002047,-0.012850
1,-0.225939,0.124835,-0.000272,-0.000090,-0.030649,-0.103516,-0.039661,-0.149412,-0.685802,-0.622436,-0.789531,-0.645444,0.190102,-0.311184,0.121082,-0.807541,0.010869,-0.014419
2,1.461891,-1.562994,-0.000273,-0.000089,-0.018006,-0.093629,-0.024912,-0.136887,1.630661,1.499467,1.812804,1.579711,-0.393579,0.546427,-0.152849,1.794796,-0.004316,-0.017770
3,-1.762054,0.940305,-0.000272,-0.000089,-0.003256,-0.074487,0.124696,-0.028337,0.124047,0.127711,0.165009,0.079417,0.010995,0.122890,-0.133885,0.147003,-0.003035,-0.012060
4,-1.212086,1.395450,-0.000271,-0.000089,-0.008525,-0.079536,0.003536,-0.119822,0.237832,0.216212,0.329366,0.167919,-0.064862,0.274605,-0.209742,0.311359,-0.002350,-0.012505
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2305,-1.799983,-0.406166,-0.000271,-0.000089,-0.012739,-0.106025,-0.020697,-0.141164,-0.318456,-0.236826,-0.363886,-0.354655,0.244888,-0.136290,-0.108598,-0.381895,-0.000869,-0.003645
2306,0.342992,-1.885389,-0.000272,-0.000089,-0.011685,-0.091065,-0.029127,-0.134703,1.717756,1.609039,1.848626,1.695604,-0.326150,0.392605,-0.066456,1.830616,-0.004884,-0.018706
2307,-0.851763,-0.975098,-0.000272,-0.000089,-0.012738,-0.089240,-0.018590,-0.134196,0.416237,0.351068,0.573798,0.323846,-0.195508,0.472678,-0.277170,0.555788,-0.002128,-0.013792
2308,-0.510404,0.181729,-0.000271,-0.000089,-0.025380,-0.105010,-0.038609,-0.150121,-0.684400,-0.622435,-0.785318,-0.645443,0.185889,-0.302754,0.116867,-0.803325,0.010869,-0.014419


In [33]:
pd.DataFrame(autoencoder_feat_eng(b.data))

,0,1,2,3,4,5,6,7
0,0.004116,-0.011230,0.042233,0.010510,0.011083,0.022492,-0.010171,0.019217
1,0.002260,-0.000263,-0.001065,0.002959,-0.005706,-0.006836,0.002429,0.005437
2,0.002559,0.001494,0.000844,-0.004319,-0.010669,-0.001595,-0.009944,-0.002775
3,0.029137,-0.002212,0.046140,0.004689,0.000075,0.006397,-0.044727,-0.033075
4,0.003178,0.007075,0.013539,-0.010819,-0.007581,-0.019420,0.008462,-0.003639
...,...,...,...,...,...,...,...,...
2305,0.008235,0.004950,-0.009224,-0.003310,0.002924,-0.005665,-0.003559,0.004043
2306,-0.004026,-0.007570,0.004335,-0.007633,0.012511,-0.005068,0.007848,-0.001441
2307,-0.010692,-0.014701,-0.000654,0.010974,0.009239,-0.011387,-0.003138,-0.014874
2308,-0.000504,0.000759,0.002134,0.003545,-0.005417,-0.004272,0.001206,-0.000586


## Function get_scores

Get the NCA score of a specific dataset using genie mst algorithm

In [34]:
import genieclust
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_scores(battery, dataset, apply_scale=False):
  b = clustbench.load_dataset(battery, dataset, url=data_url)

  X_transformed = b.data
  if apply_scale:
    # Scale the data
    scaler = StandardScaler()
    X_transformed = scaler.fit_transform(X_transformed)

  g = genieclust.Genie(n_clusters=b.n_clusters[0])
  results = clustbench.fit_predict_many(g, X_transformed, b.n_clusters)
  scores = clustbench.get_score(b.labels, results)

  return scores

## Function get_scores_with_autoencoder

Get the NCA score of a specific dataset applying the autoencoder transformation to the data and then using genie mst algorithm

In [35]:
import genieclust
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_scores_with_autoencoder(battery, dataset, apply_scale=False, n_embeddings=2):
  b = clustbench.load_dataset(battery, dataset, url=data_url)

  X_transformed = autoencoder_feat_eng(b.data, n_embeddings=n_embeddings)

  if apply_scale:
    # Scale the data
    scaler = StandardScaler()
    X_transformed = scaler.fit_transform(X_transformed)

  g = genieclust.Genie(n_clusters=b.n_clusters[0])
  results = clustbench.fit_predict_many(g, X_transformed, b.n_clusters)
  scores = clustbench.get_score(b.labels, results)

  return scores

## Execute get_scores on all datasets as baseline

In [36]:
import tqdm
import pandas as pd
columns = ['Battery', 'Dataset', 'Genie NCA Score']
df = pd.DataFrame(columns=columns)
scores_lists = {}
for col in columns:
  scores_lists[col] = []

for battery in tqdm.tqdm(battery_datasets_dict.keys(), desc="Processing Datasets"):
  for dataset in battery_datasets_dict[battery]:
    scores_lists['Battery'].append(battery)
    scores_lists['Dataset'].append(dataset)
    scores_lists['Genie NCA Score'].append(get_scores(battery, dataset))

df = pd.DataFrame.from_dict(scores_lists)

Processing Datasets: 100%|██████████| 2/2 [00:04<00:00,  2.11s/it]


In [37]:
df

,Battery,Dataset,Genie NCA Score
0,fcps,atom,1.000000
1,fcps,chainlink,1.000000
2,fcps,engytime,0.918870
3,fcps,hepta,1.000000
4,fcps,lsun,1.000000
5,fcps,target,1.000000
6,fcps,tetra,1.000000
7,fcps,twodiamonds,0.987500
8,fcps,wingnut,1.000000
9,uci,ecoli,0.435664


## Execute get_scores_with_autoencoder on all datasets trying different numbers of embeddings (last hidden layer of the neural network)



In [38]:
n_embeddings_list = [128,256,512,1024]

scores_lists = {}
for n_embs in tqdm.tqdm(n_embeddings_list, desc="Processing Datasets"):
  for battery in battery_datasets_dict.keys():
    for dataset in battery_datasets_dict[battery]:
      column_name = 'Genie+Autoencoder '+ str(n_embs) +' NCA Score'
      if column_name not in scores_lists:
        scores_lists[column_name] = []
      scores_lists[column_name].append(get_scores_with_autoencoder(battery, dataset, n_embeddings=n_embs))

df_ssnn = pd.DataFrame.from_dict(scores_lists)

Processing Datasets: 100%|██████████| 4/4 [02:41<00:00, 40.46s/it]


# Final Results

In [39]:
df = pd.concat([df, df_ssnn], axis=1)
numerical_cols = df.columns[2:]
df.style.highlight_max(axis=1, subset=numerical_cols)

,Battery,Dataset,Genie NCA Score,Genie+Autoencoder 128 NCA Score,Genie+Autoencoder 256 NCA Score,Genie+Autoencoder 512 NCA Score,Genie+Autoencoder 1024 NCA Score
0,fcps,atom,1.000000,0.372500,0.427500,0.607500,0.660000
1,fcps,chainlink,1.000000,1.000000,1.000000,1.000000,1.000000
2,fcps,engytime,0.918870,0.920332,0.952609,0.919355,0.918866
3,fcps,hepta,1.000000,0.716667,0.750000,0.833333,0.638889
4,fcps,lsun,1.000000,1.000000,1.000000,1.000000,1.000000
5,fcps,target,1.000000,1.000000,1.000000,1.000000,1.000000
6,fcps,tetra,1.000000,0.946667,0.996667,0.570000,0.553333
7,fcps,twodiamonds,0.987500,0.990000,0.990000,0.990000,0.990000
8,fcps,wingnut,1.000000,0.933071,1.000000,1.000000,1.000000
9,uci,ecoli,0.435664,0.407200,0.222399,0.558706,0.455858
